# Flow capacity vs. derived-tail collapse — GPU sweep

Does scaling the flow (or swapping affine coupling for a more expressive
**rational-quadratic spline**) recover the tail that an equation-**derived**
integrator clips, or does it plateau far below what max-entropy wants?

**Setup.** An `EquationEstimate` like
`anomaly = trend + 0.075*enso_oni - volcanic ~ N(0, 0.035)` turns `anomaly` into a
derived flow coordinate. The `eqn_dist` loss pins only the residual's first two
moments (mean 0, var σ²), **not** its independence from the right-hand side. A
finite flow satisfies those two moments while anti-correlating the residual with
the RHS in the tail — clipping `P(anomaly > 1.60)` to ≈0, even though the
max-entropy solution (independent, *same marginals*) has a ~0.06–0.09 tail **and**
higher entropy. So per flow size + family we log `P_flow` (derived tail),
`P_maxent` (reconvolved-independent target), and `I` (nats the clip sacrifices).

**All experiment logic lives in `metaculus/capacity_experiment.py`** (in the repo).
This notebook is a thin shell: the setup cell **clones-or-`git pull`s** the repo, so
iterating on the sweep is *edit `capacity_experiment.py` → push → rerun the setup
cell* — no notebook re-upload.

**Runtime → Change runtime type → GPU** (T4 is fine), then Run all.


In [ ]:
# --- Setup: clone-or-pull the repo, install deps Colab lacks, keep Colab's jax ---
import os, sys, subprocess

REPO = "/content/calibrated_response"
BRANCH = "main"   # branch carrying the equation-language + spline changes
URL = "https://github.com/amdson/calibrated_response.git"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, URL, REPO],
                   check=True)
else:
    # re-run picks up the latest payload without a re-clone (or notebook re-upload)
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH],
                   check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"],
                   check=True)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# optax/jaxopt are the only solver deps Colab doesn't ship; installing the
# package itself would drag in pinned jax/jaxlib and clobber the GPU build.
%pip -q install optax jaxopt

import jax
jax.config.update("jax_compilation_cache_dir", f"{REPO}/.jax_cache")
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
print("jax backend:", jax.default_backend(), jax.devices())
assert jax.default_backend() != "cpu", "No GPU — switch the runtime type first"

# import-guard: confirm the pulled branch has the equation language + spline flow
from calibrated_response.models.query import EquationEstimate            # noqa: F401
from calibrated_response.maxent_sampler.spline_sampler import SplineFlowSampler  # noqa: F401
from metaculus.capacity_experiment import run_sweep, plot_results, default_configs
print(f"payload OK — {len(default_configs())} configs in the default grid")


In [ ]:
# --- Sweep knobs -----------------------------------------------------------
# QUICK=True: a tiny grid to validate the whole notebook end-to-end (~minutes).
# QUICK=False: the full affine-vs-spline capacity ladder from default_configs().
QUICK = True

if QUICK:
    CONFIGS = [
        dict(flow_type="affine", n_layers=4, hidden=32, n_dummy=0),
        dict(flow_type="spline", n_layers=4, hidden=32, n_dummy=0, num_bins=8),
    ]
    SEEDS, STEPS, N_SAMPLES, EVAL = (0,), 800, 4000, 50_000
else:
    CONFIGS = default_configs()
    SEEDS, STEPS, N_SAMPLES, EVAL = (0, 1, 2), 5000, 6000, 200_000


In [ ]:
# --- Run the sweep (resumable: rows append to results/capacity_sweep.jsonl) ---
rows = run_sweep(configs=CONFIGS, seeds=SEEDS, steps=STEPS,
                 n_samples=N_SAMPLES, eval_samples=EVAL,
                 out_path="results/capacity_sweep.jsonl")
print(f"\n{len(rows)} rows total in results/capacity_sweep.jsonl")


In [ ]:
# --- Plot: does the derived tail climb to the max-ent target? (affine vs spline) ---
plot_results(rows, save="results/capacity_sweep.png");


In [ ]:
# --- Download results (unzip into results/ locally to merge) ---
import shutil
shutil.make_archive("/content/capacity_sweep_results", "zip", "results")
try:
    from google.colab import files
    files.download("/content/capacity_sweep_results.zip")
except ImportError:
    print("not on Colab — results in results/")
